In [ ]:
from pathlib import Path
import pandas as pd
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"
path = DATA_DIR / "Amazon_Reviews.csv"
dataset = pd.read_csv(path)
df = pd.DataFrame(dataset)
df.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21214 entries, 0 to 21213
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Reviewer Name       21214 non-null  object
 1   Profile Link        21163 non-null  object
 2   Country             21054 non-null  object
 3   Review Count        21055 non-null  object
 4   Review Date         21055 non-null  object
 5   Rating              21055 non-null  object
 6   Review Title        21055 non-null  object
 7   Review Text         21055 non-null  object
 8   Date of Experience  20947 non-null  object
dtypes: object(9)
memory usage: 1.5+ MB


deleting unnecessary cols and dropping nulls:

In [3]:
df = df[["Review Text", "Rating"]]
df = df.dropna()
df

,Review Text,Rating
0,"I registered on the website, tried to order a ...",Rated 1 out of 5 stars
1,Had multiple orders one turned up and driver h...,Rated 1 out of 5 stars
2,I informed these reprobates that I WOULD NOT B...,Rated 1 out of 5 stars
3,I have bought from Amazon before and no proble...,Rated 1 out of 5 stars
4,If I could give a lower rate I would! I cancel...,Rated 1 out of 5 stars
...,...,...
21209,"I have had perfect order fulfillment, and fast...",Rated 5 out of 5 stars
21210,"I have had perfect order fulfillment, and fast...",Rated 5 out of 5 stars
21211,I always find myself going back to amazon beco...,Rated 3 out of 5 stars
21212,I have placed an abundance of orders with Amaz...,Rated 5 out of 5 stars


In [4]:
df['Rating']

0        Rated 1 out of 5 stars
1        Rated 1 out of 5 stars
2        Rated 1 out of 5 stars
3        Rated 1 out of 5 stars
4        Rated 1 out of 5 stars
                  ...          
21209    Rated 5 out of 5 stars
21210    Rated 5 out of 5 stars
21211    Rated 3 out of 5 stars
21212    Rated 5 out of 5 stars
21213    Rated 4 out of 5 stars
Name: Rating, Length: 21055, dtype: object

extracting rating number out of Rating col:

In [5]:
df["Rating"] = df["Rating"].astype(str)
df["Rating"] = df["Rating"].str.extract(r'(\d)').astype(int)
df

,Review Text,Rating
0,"I registered on the website, tried to order a ...",1
1,Had multiple orders one turned up and driver h...,1
2,I informed these reprobates that I WOULD NOT B...,1
3,I have bought from Amazon before and no proble...,1
4,If I could give a lower rate I would! I cancel...,1
...,...,...
21209,"I have had perfect order fulfillment, and fast...",5
21210,"I have had perfect order fulfillment, and fast...",5
21211,I always find myself going back to amazon beco...,3
21212,I have placed an abundance of orders with Amaz...,5


converting ratings to sentiments and adding to df cols:

In [6]:
def convert_rating(rating):
    if rating >= 4:
        return 1
    elif rating <= 2:
        return 0
    else:
        return None

df["label"] = df["Rating"].apply(convert_rating)

df = df.dropna(subset=["label"])

df["label"].value_counts()

label
0.0    14350
1.0     5820
Name: count, dtype: int64

In [7]:
df

,Review Text,Rating,label
0,"I registered on the website, tried to order a ...",1,0.0
1,Had multiple orders one turned up and driver h...,1,0.0
2,I informed these reprobates that I WOULD NOT B...,1,0.0
3,I have bought from Amazon before and no proble...,1,0.0
4,If I could give a lower rate I would! I cancel...,1,0.0
...,...,...,...
21208,I buy almost everything I need at Amazon. Ser...,5,1.0
21209,"I have had perfect order fulfillment, and fast...",5,1.0
21210,"I have had perfect order fulfillment, and fast...",5,1.0
21212,I have placed an abundance of orders with Amaz...,5,1.0


In [8]:
df["label"].value_counts()

label
0.0    14350
1.0     5820
Name: count, dtype: int64

downsampling 0 labels:

In [9]:
min_size = df["label"].value_counts().min()

df = df.groupby("label").sample(min_size, random_state=42)
df["label"].value_counts()

label
0.0    5820
1.0    5820
Name: count, dtype: int64

shuffling records:

In [10]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

checking reviews lenghts:

In [11]:
print(df["Review Text"].str.len().describe())

count    11640.000000
mean       388.507216
std        471.786705
min         10.000000
25%        105.000000
50%        244.000000
75%        496.000000
max       8166.000000
Name: Review Text, dtype: float64


trucating:

In [12]:
df["Review Text"] = df["Review Text"].str[:1000]
df.rename(columns={"Review Text": "text"}, inplace=True)
df.rename(columns={"Rating": "rating"}, inplace=True)
df

,text,rating,label
0,Amazon is an exceptional company that provides...,5,1.0
1,I ordered a pair of Adidas trainers through Am...,1,0.0
2,It's amazingly one of the best in purchase and...,4,1.0
3,Amazon is not what it was! It turns out Amazon...,1,0.0
4,"Repeatedly value and service, one of first pla...",5,1.0
...,...,...,...
11635,The delivery is good too.,5,1.0
11636,I dont reciveve text or email from all pacakag...,1,0.0
11637,I have used Amazon from time to time for deliv...,1,0.0
11638,"amazon customer service is so rude, I tried ex...",1,0.0


In [13]:
df["label"] = df["label"].astype(int)
df['label']

0        1
1        0
2        1
3        0
4        1
        ..
11635    1
11636    0
11637    0
11638    0
11639    1
Name: label, Length: 11640, dtype: int32

saving the processed dataframe:

In [ ]:
df.to_csv(DATA_DIR / "dataprocessed_reviews.csv", index=False)